In [ ]:
import numpy as np
import numpy.linalg as la
import matplotlib
import matplotlib.pyplot as plt
import netCDF4 as nc4
import os.path
import random as rnd

In [ ]:
E3SM_DATA_LOC = '/Users/sant720/Data/e3sm-microphysics/'
E3SM_DATA_FILES = [
    'microphysics_control_data.eam.h1.2010-01-31-36000.nc', 'microphysics_control_data.eam.h1.2010-03-02-72000.nc',
    'microphysics_control_data.eam.h1.2010-04-02-21600.nc', 'microphysics_control_data.eam.h1.2010-05-02-57600.nc',
    'microphysics_control_data.eam.h1.2010-06-02-07200.nc', 'microphysics_control_data.eam.h1.2010-07-02-43200.nc',
    'microphysics_control_data.eam.h1.2010-08-01-79200.nc', 'microphysics_control_data.eam.h1.2010-09-01-28800.nc',
    'microphysics_control_data.eam.h1.2010-10-01-64800.nc', 'microphysics_control_data.eam.h1.2010-11-01-14400.nc',
    'microphysics_control_data.eam.h1.2010-12-01-50400.nc', 'microphysics_control_data.eam.h1.2011-01-01-00000.nc',
    'microphysics_control_data.eam.h1.2011-01-31-36000.nc', 'microphysics_control_data.eam.h1.2011-03-02-72000.nc',
]

In [ ]:
AST_IDX = 0
PMID_IDX = 3
ZI_IDX = 4
T_IDX = 5
Q_IDX = 6
QC_IDX = 7
QI_IDX = 8
NC_IDX = 9
NI_IDX = 10
QR_IDX = 11
NR_IDX = 12
PDEL_IDX = 15

In [ ]:
with nc4.Dataset(os.path.join(E3SM_DATA_LOC, E3SM_DATA_FILES[0])) as data:
    nlev = len(data.dimensions['lev'])
    nilev = len(data.dimensions['ilev'])
    ncol = len(data.dimensions['ncol'])
    nstep = len(data.dimensions['micro_step_dim'])
    rainfrac = data['CLOUDFRAC_RAIN_MICRO'][0,:,:,:]
    ps = data['PS'][0,:]
    ast = data['P3_input'][0,AST_IDX,:,:nlev,:]
    pmid = data['P3_input'][0,PMID_IDX,:,:nlev,:]
    zi = data['P3_input'][0,ZI_IDX,:,:,:]
    t = data['P3_input'][0,T_IDX,:,:nlev,:]
    q = data['P3_input'][0,Q_IDX,:,:nlev,:]
    qc = data['P3_input'][0,QC_IDX,:,:nlev,:]
    nc = data['P3_input'][0,NC_IDX,:,:nlev,:]
    qi = data['P3_input'][0,QI_IDX,:,:nlev,:]
    ni = data['P3_input'][0,NI_IDX,:,:nlev,:]
    qr = data['P3_input'][0,QR_IDX,:,:nlev,:]
    nr = data['P3_input'][0,NR_IDX,:,:nlev,:]
    pdel = data['P3_input'][0,PDEL_IDX,:,:nlev,:]

In [ ]:
plt.hist(np.log10(qc.flatten()), bins=100, range=(-10, 0))
plt.xlabel('$\log_{10}(q_c)$')
plt.ylabel('Count in first file')
plt.savefig("first_file_qc_histogram.png", dpi=300)

In [ ]:
plt.hist(np.log10(qr.flatten()), bins=100, range=(-10, 0))
plt.xlabel('$\log_{10}(q_r)$')
plt.ylabel('Count in first file')
plt.savefig("first_file_qr_histogram.png", dpi=300)

In [ ]:
np.count_nonzero(qr >= 1.e-3)

In [ ]:
qr.max()

In [ ]:
plt.hist(np.log10((qr/rainfrac).flatten()), bins=100, range=(-10, 0), label="All grid points")
plt.hist(np.where(qr.flatten() >= 1.e-6, np.log10((qr/rainfrac).flatten()), np.nan),
         bins=100, range=(-10, 0), label="Points with $q_r ≥ 10^{-6}$")
plt.hist(np.where(np.logical_and(rainfrac.flatten() <= 1.e-4, qr.flatten() >= 1.e-6), np.log10((qr/rainfrac).flatten()), np.nan),
         bins=100, range=(-10, 0), label="Points with $q_r ≥ 10^{-6}$ and clipped rain fraction")
plt.xlabel('$\log_{10}(q_r$ / rain area fraction$)$')
plt.ylabel('Count in first file')
plt.legend()
plt.savefig("first_file_in-area_qr_histogram.png", dpi=300)

In [ ]:
def find_cloud_base_idx(qc, qc_thresh):
    idx = -1
    for i, qc_loc in enumerate(qc):
        if qc_loc >= qc_thresh:
            idx = i
    return idx

In [ ]:
def cloud_base_idx_if_eligible(qc, qr, qi, zi, qc_thresh=1.e-6, qr_thresh=1.e-6, qi_thresh=1.e-6, zi_bounds=(400., 2000.)):
    idx = find_cloud_base_idx(qc, qc_thresh)
    if idx == -1 or qr[idx] < qr_thresh or not (zi_bounds[0] <= zi[idx+1] <= zi_bounds[1]):
        return -1
    if np.any(qi[idx:] > qi_thresh):
        return -1
    return idx

In [ ]:
eligible_file_nums = []
eligible_cols = []
eligible_time_steps = []
base_idxs = []
for ifile, file in enumerate(E3SM_DATA_FILES[2:]):
    with nc4.Dataset(os.path.join(E3SM_DATA_LOC, file)) as data:
        zi = data['P3_input'][0,ZI_IDX,:,:,:]
        qc = data['P3_input'][0,QC_IDX,:,:nlev,:]
        qi = data['P3_input'][0,QI_IDX,:,:nlev,:]
        qr = data['P3_input'][0,QR_IDX,:,:nlev,:]
        for step in range(nstep):
            for icol in range(ncol):
                idx = cloud_base_idx_if_eligible(qc[step,:,icol], qr[step,:,icol], qi[step,:,icol], zi[step,:,icol])
                if idx != -1:
                    eligible_file_nums.append(ifile)
                    eligible_cols.append(icol)
                    eligible_time_steps.append(step)
                    base_idxs.append(idx)
eligible_file_nums = np.array(eligible_file_nums, dtype='int32')
eligible_cols = np.array(eligible_cols, dtype='int32')
eligible_time_steps = np.array(eligible_time_steps, dtype='int32')
base_idxs = np.array(base_idxs, dtype='int32')

In [ ]:
nsamp = len(eligible_cols)
min_base_idx = min(base_idxs)
max_levs = nlev - min_base_idx

In [ ]:
print(nsamp, min_base_idx, max_levs)

In [ ]:
ALL_SAMPLES_FILE = 'all_rainshaft_samples.nc'

In [ ]:
with nc4.Dataset(os.path.join(E3SM_DATA_LOC, ALL_SAMPLES_FILE), 'w') as out_data:
    samp_dim = out_data.createDimension('nsamp', nsamp)
    lev_dim = out_data.createDimension('nlev', max_levs)
    ilev_dim = out_data.createDimension('nilev', max_levs+1)
    file_dim = out_data.createDimension('file', len(E3SM_DATA_FILES[2:]))
    file_name_length_dim = out_data.createDimension('file_name_length', 128)
    file_names = out_data.createVariable('source_file_names', 'c', (file_dim, file_name_length_dim))
    for i, file in enumerate(E3SM_DATA_FILES[2:]):
        file_names[i,:len(file)] = file
    file_names.description = "Name of file associated with each file_num value"
    file_num = out_data.createVariable('file_num', 'i4', (samp_dim,))
    file_num[:] = eligible_file_nums
    file_num.description = "Number representing which of the files in 'source_file_names' the sample is taken from"
    col_num = out_data.createVariable('col_num', 'i4', (samp_dim,))
    col_num[:] = eligible_cols
    col_num.description = "Column number of sample on the given file"
    time_step_num = out_data.createVariable('time_step_num', 'i4', (samp_dim,))
    time_step_num[:] = eligible_time_steps
    time_step_num.description = "Microphysics substep number from which sample data was taken"
    nlev_samp = out_data.createVariable('nlev_samp', 'i4', (samp_dim,))
    nlev_samp[:] = nlev - base_idxs
    nlev_samp.description = "Number of rainshaft levels for this sample"
    rainfrac_out = out_data.createVariable('rainfrac', 'f8', (samp_dim, lev_dim))
    rainfrac_out.description = "Rain fraction"
    rainfrac_out.units = '1'
    ps_out = out_data.createVariable('surface_pressure', 'f8', (samp_dim,))
    ps_out.description = "Surface pressure"
    ps_out.units = 'Pa'
    cloudfrac_out = out_data.createVariable('cloudfrac', 'f8', (samp_dim, lev_dim))
    cloudfrac_out.description = "Cloud fraction"
    cloudfrac_out.units = '1'
    pmid_out = out_data.createVariable('pmid', 'f8', (samp_dim, lev_dim))
    pmid_out.description = "Mid-level pressure"
    pmid_out.units = 'Pa'
    zi_out = out_data.createVariable('zi', 'f8', (samp_dim, ilev_dim))
    zi_out.description = "Altitude at level interfaces"
    zi_out.units = 'm'
    t_out = out_data.createVariable('t', 'f8', (samp_dim, lev_dim))
    t_out.description = "Temperature"
    t_out.units = 'K'
    q_out = out_data.createVariable('q', 'f8', (samp_dim, lev_dim))
    q_out.description = "Humidity"
    q_out.units = 'kg/kg'
    qc_out = out_data.createVariable('qc', 'f8', (samp_dim, lev_dim))
    qc_out.description = "Cloud liquid mass mixing ratio"
    qc_out.units = 'kg/kg'
    nc_out = out_data.createVariable('nc', 'f8', (samp_dim, lev_dim))
    nc_out.description = "Cloud liquid drop number concentration"
    nc_out.units = '1/kg'
    qr_out = out_data.createVariable('qr', 'f8', (samp_dim, lev_dim))
    qr_out.description = "Rain mass mixing ratio"
    qr_out.units = 'kg/kg'
    nr_out = out_data.createVariable('nr', 'f8', (samp_dim, lev_dim))
    nr_out.description = "Rain drop number concentration"
    nr_out.units = '1/kg'
    pdel_out = out_data.createVariable('pdel', 'f8', (samp_dim, lev_dim))
    pdel_out.description = "Pressure difference between bottom and top of level"
    pdel_out.units = 'Pa'
    sidx = 0
    for ifile, file in enumerate(E3SM_DATA_FILES[2:]):
        with nc4.Dataset(os.path.join(E3SM_DATA_LOC, file)) as data:
            while sidx < nsamp and ifile == eligible_file_nums[sidx]:
                nlev_out = nlev_samp[sidx]
                col = col_num[sidx]
                step = time_step_num[sidx]
                base_idx = base_idxs[sidx]
                rainfrac_out[sidx,:nlev_out] = data['CLOUDFRAC_RAIN_MICRO'][0,step,base_idx:,col]
                ps_out[sidx] = data['PS'][0,col]
                cloudfrac_out[sidx,:nlev_out] = data['P3_input'][0,AST_IDX,step,base_idx:nlev,col]
                pmid_out[sidx,:nlev_out] = data['P3_input'][0,PMID_IDX,step,base_idx:nlev,col]
                zi_out[sidx,:nlev_out+1] = data['P3_input'][0,ZI_IDX,step,base_idx:,col]
                t_out[sidx,:nlev_out] = data['P3_input'][0,T_IDX,step,base_idx:nlev,col]
                q_out[sidx,:nlev_out] = data['P3_input'][0,Q_IDX,step,base_idx:nlev,col]
                qc_out[sidx,:nlev_out] = data['P3_input'][0,QC_IDX,step,base_idx:nlev,col]
                nc_out[sidx,:nlev_out] = data['P3_input'][0,NC_IDX,step,base_idx:nlev,col]
                qr_out[sidx,:nlev_out] = data['P3_input'][0,QR_IDX,step,base_idx:nlev,col]
                nr_out[sidx,:nlev_out] = data['P3_input'][0,NR_IDX,step,base_idx:nlev,col]
                pdel_out[sidx,:nlev_out] = data['P3_input'][0,PDEL_IDX,step,base_idx:nlev,col]
                sidx += 1

In [ ]:
RANDOM_SAMPLES_FILE = 'random_rainshaft_samples.nc'

In [ ]:
SAMPLES_NEEDED = 1000
sidx_out = 0
samp_list = []
with nc4.Dataset(os.path.join(E3SM_DATA_LOC, RANDOM_SAMPLES_FILE), 'w') as out_data:
    samp_dim = out_data.createDimension('nsamp', SAMPLES_NEEDED)
    lev_dim = out_data.createDimension('nlev', max_levs)
    ilev_dim = out_data.createDimension('nilev', max_levs+1)
    file_dim = out_data.createDimension('file', len(E3SM_DATA_FILES[2:]))
    file_name_length_dim = out_data.createDimension('file_name_length', 128)
    file_names = out_data.createVariable('source_file_names', 'c', (file_dim, file_name_length_dim))
    for i, file in enumerate(E3SM_DATA_FILES[2:]):
        file_names[i,:len(file)] = file
    file_names.description = "Name of file associated with each file_num value"
    file_num = out_data.createVariable('file_num', 'i4', (samp_dim,))
    file_num.description = "Number representing which of the files in 'source_file_names' the sample is taken from"
    col_num = out_data.createVariable('col_num', 'i4', (samp_dim,))
    col_num.description = "Column number of sample on the given file"
    time_step_num = out_data.createVariable('time_step_num', 'i4', (samp_dim,))
    time_step_num.description = "Microphysics substep number from which sample data was taken"
    nlev_samp = out_data.createVariable('nlev_samp', 'i4', (samp_dim,))
    nlev_samp.description = "Number of rainshaft levels for this sample"
    rainfrac_out = out_data.createVariable('rainfrac', 'f8', (samp_dim, lev_dim))
    rainfrac_out.description = "Rain fraction"
    rainfrac_out.units = '1'
    ps_out = out_data.createVariable('surface_pressure', 'f8', (samp_dim,))
    ps_out.description = "Surface pressure"
    ps_out.units = 'Pa'
    cloudfrac_out = out_data.createVariable('cloudfrac', 'f8', (samp_dim, lev_dim))
    cloudfrac_out.description = "Cloud fraction"
    cloudfrac_out.units = '1'
    pmid_out = out_data.createVariable('pmid', 'f8', (samp_dim, lev_dim))
    pmid_out.description = "Mid-level pressure"
    pmid_out.units = 'Pa'
    zi_out = out_data.createVariable('zi', 'f8', (samp_dim, ilev_dim))
    zi_out.description = "Altitude at level interfaces"
    zi_out.units = 'm'
    t_out = out_data.createVariable('t', 'f8', (samp_dim, lev_dim))
    t_out.description = "Temperature"
    t_out.units = 'K'
    q_out = out_data.createVariable('q', 'f8', (samp_dim, lev_dim))
    q_out.description = "Humidity"
    q_out.units = 'kg/kg'
    qc_out = out_data.createVariable('qc', 'f8', (samp_dim, lev_dim))
    qc_out.description = "Cloud liquid mass mixing ratio"
    qc_out.units = 'kg/kg'
    nc_out = out_data.createVariable('nc', 'f8', (samp_dim, lev_dim))
    nc_out.description = "Cloud liquid drop number concentration"
    nc_out.units = '1/kg'
    qr_out = out_data.createVariable('qr', 'f8', (samp_dim, lev_dim))
    qr_out.description = "Rain mass mixing ratio"
    qr_out.units = 'kg/kg'
    nr_out = out_data.createVariable('nr', 'f8', (samp_dim, lev_dim))
    nr_out.description = "Rain drop number concentration"
    nr_out.units = '1/kg'
    pdel_out = out_data.createVariable('pdel', 'f8', (samp_dim, lev_dim))
    pdel_out.description = "Pressure difference between bottom and top of level"
    pdel_out.units = 'Pa'
    while sidx_out < SAMPLES_NEEDED:
        sidx = rnd.randrange(nsamp)
        if sidx in samp_list:
            continue
        ifile = eligible_file_nums[sidx]
        col = eligible_cols[sidx]
        step = eligible_time_steps[sidx]
        base_idx = base_idxs[sidx]
        nlev_out = nlev - base_idx
        file_num[sidx_out] = ifile
        col_num[sidx_out] = col
        time_step_num[sidx_out] = step
        nlev_samp[sidx_out] = nlev_out
        with nc4.Dataset(os.path.join(E3SM_DATA_LOC, E3SM_DATA_FILES[ifile+2])) as data:
            rainfrac_out[sidx_out,:nlev_out] = data['CLOUDFRAC_RAIN_MICRO'][0,step,base_idx:,col]
            ps_out[sidx_out] = data['PS'][0,col]
            cloudfrac_out[sidx_out,:nlev_out] = data['P3_input'][0,AST_IDX,step,base_idx:nlev,col]
            pmid_out[sidx_out,:nlev_out] = data['P3_input'][0,PMID_IDX,step,base_idx:nlev,col]
            zi_out[sidx_out,:nlev_out+1] = data['P3_input'][0,ZI_IDX,step,base_idx:,col]
            t_out[sidx_out,:nlev_out] = data['P3_input'][0,T_IDX,step,base_idx:nlev,col]
            q_out[sidx_out,:nlev_out] = data['P3_input'][0,Q_IDX,step,base_idx:nlev,col]
            qc_out[sidx_out,:nlev_out] = data['P3_input'][0,QC_IDX,step,base_idx:nlev,col]
            nc_out[sidx_out,:nlev_out] = data['P3_input'][0,NC_IDX,step,base_idx:nlev,col]
            qr_out[sidx_out,:nlev_out] = data['P3_input'][0,QR_IDX,step,base_idx:nlev,col]
            nr_out[sidx_out,:nlev_out] = data['P3_input'][0,NR_IDX,step,base_idx:nlev,col]
            pdel_out[sidx_out,:nlev_out] = data['P3_input'][0,PDEL_IDX,step,base_idx:nlev,col]
        samp_list.append(sidx)
        sidx_out += 1